### Exercise 3: date and time MCP server

`get_current_date` returns the current local date as `YYYY-MM-DD`

`get_current_datetime` returns the current local date and time as `YYYY-MM-DDTHH:MM:SS`

The server runs at `http://127.0.0.1:8002/mcp`.

In [1]:
import datetime
import re
import socket
import threading
import time

from fastmcp import FastMCP
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

### Define the MCP server and tools

In [2]:
mcp = FastMCP("Current date and time")


@mcp.tool(description="Get the current local date in YYYY-MM-DD format.")
def get_current_date() -> str:
    return datetime.date.today().isoformat()


@mcp.tool(
    description=(
        "Get the current local date and time in ISO 8601 format, "
        "with precision up to seconds: YYYY-MM-DDTHH:MM:SS."
    )
)
def get_current_datetime() -> str:
    return datetime.datetime.now().replace(microsecond=0).isoformat()

### Start the server on port 8002

In [3]:
HOST = "127.0.0.1"
PORT = 8002
MCP_URL = f"http://{HOST}:{PORT}/mcp"


def port_is_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as connection:
        connection.settimeout(0.2)
        return connection.connect_ex((host, port)) == 0


if port_is_open(HOST, PORT):
    print(f"A server is already running at {MCP_URL}")
else:
    server_thread = threading.Thread(
        target=mcp.run,
        kwargs={
            "transport": "streamable-http",
            "host": HOST,
            "port": PORT,
            "show_banner": False,
        },
        daemon=True,
        name="exercise3-mcp-server",
    )
    server_thread.start()

    for _ in range(50):
        if port_is_open(HOST, PORT):
            break
        time.sleep(0.1)
    else:
        raise RuntimeError("The MCP server did not start on port 8002.")

    print(f"MCP server started at {MCP_URL}")

[06/12/26 21:19:01] INFO     Starting MCP server 'Current date and time' with transport              ]8;id=89212;file:///Users/weronika/Desktop/AGH_ISSI/IPUM/Lab6/.venv/lib/python3.11/site-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=716861;file:///Users/weronika/Desktop/AGH_ISSI/IPUM/Lab6/.venv/lib/python3.11/site-packages/fastmcp/server/server.py#2585\2585]8;;\
                             'streamable-http' on http://127.0.0.1:8002/mcp                                        

INFO:     Started server process [3162]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)


MCP server started at http://127.0.0.1:8002/mcp


### Verify the tools through an MCP client

In [4]:
async with streamable_http_client(MCP_URL) as streams:
    read_stream, write_stream, _ = streams
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        tools_response = await session.list_tools()
        tool_names = [tool.name for tool in tools_response.tools]
        print("Available tools:", tool_names)

        date_response = await session.call_tool("get_current_date")
        datetime_response = await session.call_tool("get_current_datetime")

        current_date = date_response.structuredContent["result"]
        current_datetime = datetime_response.structuredContent["result"]

        print("get_current_date result:", current_date)
        print("get_current_datetime result:", current_datetime)

INFO:     127.0.0.1:50106 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50107 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50108 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50109 - "POST /mcp HTTP/1.1" 200 OK
Available tools: ['get_current_date', 'get_current_datetime']
INFO:     127.0.0.1:50110 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50111 - "POST /mcp HTTP/1.1" 200 OK
get_current_date result: 2026-06-12
get_current_datetime result: 2026-06-12T21:20:16
INFO:     127.0.0.1:50112 - "DELETE /mcp HTTP/1.1" 200 OK


### Validate the required formats

In [5]:
date_pattern = r"\d{4}-\d{2}-\d{2}"
datetime_pattern = r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"

assert set(tool_names) == {"get_current_date", "get_current_datetime"}
assert re.fullmatch(date_pattern, current_date)
assert re.fullmatch(datetime_pattern, current_datetime)

print("Both tools are available and return the required formats.")

Both tools are available and return the required formats.


### Results and conclusions

The MCP client discovered both required tools: `get_current_date` and `get_current_datetime`. The first tool returned the local date in `YYYY-MM-DD` format, while the second returned the local date and time in ISO 8601 format up to seconds. The regular-expression checks confirmed that both returned values matched the required formats.